# GPT-5-разметка real + synthetic корзинки

Этот ноутбук заново размечает через GPT-5 отзывы, собранные на предыдущем этапе: реальные примеры + синтетические примеры по классам.

## Зачем нужен ноутбук

На предыдущем этапе была собрана корзинка отзывов по классам:

```text
data/labeled/wb_feedbacks_by_class_with_synthetic/
```

В ней лежат реальные и синтетические отзывы, сгруппированные по классам.  
Этот ноутбук повторно прогоняет все эти отзывы через более сильную модель GPT-5, чтобы получить более качественную и единообразную разметку.

## Что делает ноутбук

1. Загружает CSV-файлы из папки с real + synthetic отзывами.
2. Берет текст каждого отзыва.
3. Отправляет отзывы в GPT-5 для multi-label классификации.
4. Получает новые labels по единой системе MVP-классов.
5. Сохраняет итоговый размеченный датасет для обучения модели.

## Используемая модель

Для разметки используется модель:

```text
gpt-5
```

Модель вызывается через OpenAI API.

Ссылка на документацию OpenAI по моделям:

```text
https://platform.openai.com/docs/models
```

## Выходные данные

Итоговая папка:

```text
data/labeled/wb_feedbacks_ChatGpt_markup_from_synthetic_gpt5_V_2/
```

Основные выходные файлы:

```text
chatgpt_labeled_reviews_mvp_combined.csv
chatgpt_labeled_reviews_mvp_for_training.csv
```

Файл `chatgpt_labeled_reviews_mvp_for_training.csv` используется дальше как основной train-датасет для обучения классификатора.

## Роль в пайплайне

```text
real + synthetic отзывы по классам
        ↓
ChatGpt_markup_gpt5_from_synthetic.ipynb
        ↓
GPT-5-размеченный train-датасет
        ↓
обучение multi-label классификатора
```

На этом этапе модель-классификатор еще не обучается.  
Ноутбук только формирует качественную обучающую выборку с помощью GPT-5-разметки.

In [11]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# В Google Colab раскомментируй установку зависимостей:
# !pip install -q openai pandas tqdm pyarrow openpyxl

import os
import json
import time
import re
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI


## 1. Основные настройки

Запускать ноутбук лучше из корня проекта, где находится папка `labeled`.

Если ноутбук лежит в другом месте, измени `PROJECT_ROOT`.


In [ ]:
# ====== ПУТИ ======

PROJECT_ROOT = Path("/content/drive/MyDrive/MLops_project/project/data").resolve()
LABELED_DIR = PROJECT_ROOT / "labeled"

# ВАЖНО: берем уже готовые данные ПОСЛЕ добавления синтетики.
# Синтетика заново не генерируется.
INPUT_BY_CLASS_DIR = LABELED_DIR / "wb_feedbacks_by_class_with_synthetic"

# Новая папка с результатами повторной ChatGPT-разметки.
OUTPUT_DIR = LABELED_DIR / "wb_feedbacks_ChatGpt_markup_from_synthetic_gpt5_V_2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Новая папка по классам: те же тексты, но классы уже по new_labels.
BY_CLASS_OUTPUT_DIR = LABELED_DIR / "wb_feedbacks_by_class_with_synthetic_gpt5_relabelled_V_2"
BY_CLASS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_OUTPUT_PATH = OUTPUT_DIR / "chatgpt_labeled_reviews_mvp_combined.csv"
COMBINED_PROGRESS_JSONL_PATH = OUTPUT_DIR / "chatgpt_labeled_reviews_mvp_progress.jsonl"
TRAIN_OUTPUT_PATH = OUTPUT_DIR / "chatgpt_labeled_reviews_mvp_for_training.csv"
NEEDS_REVIEW_OUTPUT_PATH = OUTPUT_DIR / "chatgpt_labeled_reviews_mvp_needs_review.csv"

TEXT_COLUMN = "отзыв"
OLD_LABELS_COLUMN = "old_labels"
NEW_LABELS_COLUMN = "new_labels"

# ====== OPENAI ======

MODEL = "gpt-5"
BATCH_SIZE = 2
SLEEP_SECONDS = 1
MAX_COMPLETION_TOKENS = 2000
MAX_REVIEWS = None  # например 100 для теста; None — разметить все

# Ключ лучше хранить в переменных окружения / Colab Secrets.
# При необходимости временно раскомментируй и вставь свой ключ:
os.environ["OPENAI_API_KEY"] = ""

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_BY_CLASS_DIR:", INPUT_BY_CLASS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("BY_CLASS_OUTPUT_DIR:", BY_CLASS_OUTPUT_DIR)
print("MODEL:", MODEL)


PROJECT_ROOT: /content/drive/MyDrive/MLops_project/project/data
INPUT_BY_CLASS_DIR: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic
OUTPUT_DIR: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_ChatGpt_markup_from_synthetic_gpt5_V_2
BY_CLASS_OUTPUT_DIR: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic_gpt5_relabelled_V_2
MODEL: gpt-5


In [14]:
ALLOWED_LABELS_MVP = [
    "Проблема с качеством товара",
    "Проблема с размером / посадкой",
    "Несоответствие карточке товара",
    "Проблема с комплектацией / упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Проблема с возвратом",
    "Положительный / нейтральный отзыв",
    "Другая проблема",
]

ALLOWED_LABELS_SET = set(ALLOWED_LABELS_MVP)


## 2. Промпт для новой разметки

Модель получает только текст отзыва.  
Старые `labels` и `evidence` в промпт не передаются.


In [15]:
SYSTEM_PROMPT = """
Ты ассистент для разметки отзывов покупателей о товарах на маркетплейсе.

Нужно выбрать labels для каждого отзыва.

Это multi-label классификация:

* один отзыв может иметь несколько labels;
* выбирай несколько labels только если в отзыве явно есть несколько разных проблем;
* не выбирай label только по отдельному слову-триггеру;
* размечай по смыслу жалобы, а не по формальным словам;
* если конкретной жалобы нет, выбирай только "Положительный / нейтральный отзыв";
* если отзыв содержит конкретную проблему, НЕ выбирай "Положительный / нейтральный отзыв";
* если жалоба есть, но она не подходит ни под один основной класс, выбирай "Другая проблема";
* если уже выбран конкретный проблемный label, НЕ добавляй "Другая проблема";
* не используй классы вне разрешенного списка;
* не добавляй объяснения, evidence, confidence, markdown или любой текст вне JSON.

Разрешенные labels:

1. "Проблема с качеством товара"
   Брак, дефект, поломка, повреждение самого товара, плохой материал, запах, плохой пошив, царапины, сколы, трещины, хлипкость, подделка, плохая сборка, плохое состояние товара, товар не выполняет свою базовую функцию.

Ставь этот label, если:

* товар сломан, разбит, погнут, потерт, поцарапан, треснул, сколот, деформирован;
* товар выглядит как использованный, испорченный, "раздербаненный", "убитый", "покоцанный";
* материал плохой, тонкий, неприятный, хлипкий, колется, рвется, отваливается;
* товар не работает или не выполняет основную функцию: лента не держится, пузыри не получаются, открывашка гнется, пудра выпадает из футляра;
* отзыв коротко говорит, что товар бесполезный, ужасный, деньги на ветер, если из смысла понятно, что товар плохой или не работает.

Важно:

* "Деньги на ветер" само по себе не означает "Цена / ценность"; если причина в том, что товар плохой или не работает, ставь "Проблема с качеством товара".
* "Бесполезная трата денег и времени ожидания" без конкретной цены или доставки чаще относится к плохому качеству товара, а не к цене или доставке.
* Если товар не выполняет базовую функцию, это "Проблема с качеством товара", а не "Несоответствие карточке товара", если нет явного сравнения с карточкой.
* Если материал просто не совпал с заявленным или ожидаемым, это "Несоответствие карточке товара", а не "Проблема с качеством товара".
* Если главный смысл жалобы в том, что материал оказался не тем, который ожидался или был указан в карточке, не добавляй "Проблема с качеством товара" только из-за слова "синтетика", "неприятно" или "не такая ткань".
* Добавляй "Проблема с качеством товара" к несоответствию по материалу только если есть отдельная явная жалоба на качество материала как самостоятельную проблему: материал рвется, колется, вызывает сильный дискомфорт, ткань плохая/хлипкая/непригодная, качество материала плохое или "не дотягивает".
* Не добавляй "Проблема с качеством товара" только потому, что покупатель возвращает товар из-за того, что материал оказался не тем, который был заявлен. Если смысл в том, что "указано хлопок, а пришла синтетика", это "Несоответствие карточке товара".
* Если покупатель прямо пишет "качество могло быть лучше", "качество материала не дотягивает", "качество не соответствует ожиданиям", "качество так себе", это "Проблема с качеством товара".
* "Ожидала хлопковую ткань, а пришла синтетика какая-то, ужасно неприятно на ощупь." → ["Несоответствие карточке товара"]
  Причина: главный смысл — материал не совпал с ожиданием/карточкой; качество отдельно не добавляем.
* "... возврат потому, что 100% синтетика ... хотя указано хлопок ..." → ["Несоответствие карточке товара"]
  Причина: главный смысл — состав не соответствует заявленному; возврат здесь просто действие покупателя, а не отдельная проблема качества или возврата.

2. "Проблема с размером / посадкой"
   Маломерит, большемерит, не подошел размер, плохо сидит, слишком узкое/широкое/короткое/длинное, не подходит по форме, посадке, габаритам или совместимости.

Ставь этот label, если:

* одежда/обувь не подошла по размеру или посадке;
* товар не подходит по форме или габаритам;
* аксессуар/деталь не совместимы с нужным предметом.

3. "Несоответствие карточке товара"
   Товар отличается от фото, описания, характеристик, состава, цвета, количества, модели, варианта, комплектации, заявленного материала, оригинальности или заявленного эффекта.

Ставь этот label, если:

* привезли не тот товар;
* заказали один вариант, а привезли другой;
* заказали один цвет/модель/тип/артикул/название, а пришел другой;
* товар отличается от фото или описания;
* заявлена нержавейка, оригинал, конкретный состав или материал, но покупатель пишет, что это неправда;
* покупатель прямо пишет "подделка", "не оригинал", "обман", если смысл в том, что товар не соответствует заявленному;
* ожидался один материал/состав, а пришел другой: ожидала хлопок, а пришла синтетика; думала будет хлопок, а пришла синтетика; указано хлопок, а фактически 100% синтетика; заявлена кожа, а пришел кожзам;
* покупатель пишет, что материал, состав или ткань не соответствуют заявленному, ожидаемому или указанному в карточке;
* покупатель пишет, что карточка/фото/ракурс вводит в заблуждение: товар показан не той стороной, выгодным ракурсом, без понятного масштаба, из-за чего размер/вид товара воспринимается неправильно;
* покупатель пишет "не вводите покупателей в заблуждение", "показывайте лицевой стороной", "на фото выглядит иначе", если смысл в том, что карточка создала неверное ожидание о товаре.

Важно:

* "Привезли не тот наполнитель: заказывала 'Сенсацию свежести', привезли 'Классик'" → это "Несоответствие карточке товара".
* "Может специально привозят не тот товар" → это "Несоответствие карточке товара".
* Если товар просто плохой, но нет явного отличия от карточки/фото/описания/заказанного варианта, не ставь этот label.
* Если товар не работает как должен, но нет указания на обещания в карточке, это обычно "Проблема с качеством товара".
* Если в отзыве есть противопоставление "ожидала/думала/заказывала/указано/заявлено X, а пришло Y", это "Несоответствие карточке товара".
* Если главный смысл жалобы в том, что материал не тот, который ожидался или был заявлен, ставь "Несоответствие карточке товара".
* Если покупатель пишет "ожидала хлопок, а пришла синтетика", "думала будет хлопок, а это синтетика", "указано хлопок, а пришла синтетика" — основной label "Несоответствие карточке товара".
* Не добавляй "Проблема с качеством товара" только из-за слова "синтетика", если главный смысл в том, что материал не совпал с ожиданием/описанием/карточкой.
* Если покупатель пишет, что материал не соответствует заявленному, ожидаемому или указанному в карточке, по умолчанию ставь только "Несоответствие карточке товара".
* Не добавляй "Проблема с качеством товара" только потому, что покупатель возвращает товар из-за несоответствия материала.
* Добавляй "Проблема с качеством товара" дополнительно только если есть отдельная явная жалоба на качество материала: материал рвется, колется, вызывает сильный дискомфорт, ткань плохая/хлипкая/непригодная, качество материала плохое или "не дотягивает".

4. "Проблема с комплектацией / упаковкой"
   Неполный комплект, не хватает деталей или товара, неправильное/поврежденное вложение, нет инструкции, плохая инструкция, упаковка повреждена, коробка мятая, порванная или плохо упакована.

Ставь этот label, если:

* не хватает товара, детали, аксессуара, инструкции;
* инструкция есть, но с ней есть проблема: она непонятная, слишком мелкая, плохо читается, плохо объясняет использование, не хватает картинок, схем или объяснений;
* жалоба относится к инструкции, руководству, схеме сборки или объяснениям на коробке/в комплекте;
* из упаковки что-то вырезали, достали, украли или оно отсутствует;
* упаковка вскрыта, нарушена, порвана, помята, повреждена;
* коробка в плохом состоянии;
* товар не подходит для подарка из-за состояния коробки;
* этикетки наклеены плохо, оторваны или повреждены;
* товар плохо упакован.

Важно:

* "Коробка в хлам" → это "Проблема с комплектацией / упаковкой".
* "Коробка в плачевном состоянии, для подарка не подошла" → это "Проблема с комплектацией / упаковкой".
* "В ПВЗ обнаружили, что ножниц нет, вырезали их" → это "Проблема с комплектацией / упаковкой".
* Если упаковка нарушена и сам товар из-за этого выглядит поврежденным/раздербаненным, можно дополнительно поставить "Проблема с качеством товара".
* Проблемы с инструкцией относятся к "Проблема с комплектацией / упаковкой", если покупатель жалуется на отсутствие инструкции, непонятную инструкцию, мелкий текст инструкции, отсутствие картинок/схем/объяснений или сложность разобраться по инструкции.
* "Инструкция вообще непонятная, не хватает картинок и объяснений." → ["Проблема с комплектацией / упаковкой"]
* "Инструкция на коробке написана мелко и непонятно, пришлось разбираться долго." → ["Проблема с комплектацией / упаковкой"]

5. "Проблема доставки / получения"
   Долгая доставка, задержка, перенос срока, курьер, пункт выдачи, склад, потеря заказа, отмена заказа, проблема с получением, неправильное вложение на этапе доставки/получения.

Ставь этот label, если:

* доставка задержалась или срок перенесли;
* заказ потеряли или отменили;
* есть жалоба на курьера;
* есть жалоба на ПВЗ;
* есть жалоба на склад;
* проблема обнаружена при получении и покупатель прямо связывает ее со службой доставки, ПВЗ, курьером или складом;
* неправильное вложение связано с доставкой/складом/ПВЗ/курьером.

Важно:

* Не ставь этот label только потому, что в отзыве есть слова "доставка", "привезли", "пришло", "получила".
* "Сам чемодан супер, но доставка, коробка в хлам" → только "Проблема с комплектацией / упаковкой", потому что конкретная жалоба про коробку, а не про срок, курьера, ПВЗ или потерю.
* "Вот такая служба доставки" при отсутствующем товаре/вырезанном вложении → ставь "Проблема доставки / получения", потому что покупатель явно связывает проблему со службой доставки.
* "Доставка быстрая" не является проблемой доставки.

6. "Цена / ценность"
   Дорого, цена завышена, не стоит своих денег, плохое соотношение цены и качества, качество не соответствует цене, ожидал больше за эти деньги.

Ставь этот label, если:

* покупатель явно жалуется на высокую цену;
* покупатель пишет, что товар не стоит своих денег;
* покупатель сравнивает слабое качество с высокой ценой;
* есть смысл "для такой цены товар слишком плохой".

Важно:

* Не ставь этот label при любой фразе про деньги.
* "Деньги на ветер" часто означает, что товар плохой/не работает, а не что цена завышена.
* "Бесполезная трата денег" без жалобы на цену → обычно "Проблема с качеством товара".
* "Переплатили" из-за неправильного товара или платного возврата → не "Цена / ценность", если нет жалобы на обычную цену товара.
* "Платный возврат", "сняли деньги за возврат", "возврат почти 500₽" → это "Проблема с возвратом", а не "Цена / ценность".
* "Тонкий пластик за большие деньги. Возврат" → "Цена / ценность"; "Проблема с возвратом" не нужна, потому что слово "возврат" просто обозначает решение вернуть товар.
* Если покупатель пишет, что товар маленький/меньше ожидаемого и "не стоит своих денег", "в магазине дешевле", "намного дешевле" — ставь "Цена / ценность". Если при этом он жалуется, что карточка/фото ввели в заблуждение, дополнительно ставь "Несоответствие карточке товара".
* Если покупатель пишет, что "качество материала не дотягивает до заплаченной суммы", "качество не соответствует цене", "за эти деньги качество плохое" — ставь оба labels: "Проблема с качеством товара" и "Цена / ценность".
* Если есть только "за такую сумму" без явной жалобы на завышенную цену или невыгодность, не обязательно ставить "Цена / ценность"; смотри на основную жалобу.

7. "Проблема с возвратом"
   Жалобы на возврат товара или денег: не приняли возврат, отказали в возврате, сложно оформить возврат, долго возвращают деньги, не пришли деньги, списали деньги за возврат, дорогой возврат, платный возврат, удержание денег при отказе/возврате, потеря процента выкупа.

Ставь этот label, если:

* покупатель жалуется, что должен платить за возврат;
* покупатель жалуется на стоимость возврата;
* покупатель пишет "платный возврат", "возврат платный", "возврат почти 500 рублей";
* списали деньги за возврат;
* удержали деньги при возврате или отказе;
* не вернули деньги;
* возврат не приняли;
* возврат затруднен;
* покупатель говорит, что из-за ошибки продавца, склада, ПВЗ, курьера или доставки он не должен платить возврат;
* покупатель пишет, что из-за неправильного товара, неправильного вложения, поврежденного товара или плохой упаковки он теряет деньги;
* покупатель пишет про процент выкупа, снятие процента выкупа, платный отказ или удержание денег из-за неправильного товара.

Очень важно:

* Если покупатель жалуется, что из-за неправильного товара, ошибки продавца, ошибки доставки, поврежденного товара или плохой упаковки он должен платить возврат, теряет деньги, теряет процент выкупа или с него что-то списывают — обязательно ставь "Проблема с возвратом".
* Это правило работает даже если основная проблема относится к другому классу.
* Если привезли не тот товар и из-за этого покупатель пишет про возврат, процент выкупа, списание денег или платный отказ — ставь сразу два labels: "Несоответствие карточке товара" и "Проблема с возвратом".
* Если повреждена коробка/упаковка и покупатель жалуется на платный возврат — ставь сразу два labels: "Проблема с комплектацией / упаковкой" и "Проблема с возвратом".
* Если товар сломан/разбит/поврежден и покупатель жалуется на списание денег за возврат — ставь сразу два labels: "Проблема с качеством товара" и "Проблема с возвратом".

Не ставь этот label, если:

* в отзыве просто есть слово "возврат";
* покупатель просто пишет "вернула", "отправила обратно", "оформила возврат";
* "возврат" используется только как действие покупателя, без жалобы на платность, деньги, отказ, удержание, компенсацию или процент выкупа;
* покупатель пишет, что товар "подлежит возврату", "буду возвращать", "оформила возврат", но не жалуется на сам процесс возврата, платность, отказ, удержание денег или фактическое списание;
* покупатель только предполагает возможное списание денег: "надеюсь 100 р. не снимут", "надеюсь не удержат", без факта списания или жалобы на платный возврат.

Примеры:

* "Привезли не тот товар, чтобы снять процент выкупа" → ["Несоответствие карточке товара", "Проблема с возвратом"]
* "Привезли не тот наполнитель. Почему я должна платить возврат из-за вашей ошибки" → ["Несоответствие карточке товара", "Проблема с возвратом"]
* "Коробка в плохом состоянии, а возврат почти 500 рублей" → ["Проблема с комплектацией / упаковкой", "Проблема с возвратом"]
* "Сняли деньги за возврат поврежденного товара" → ["Проблема с качеством товара", "Проблема с возвратом"]
* "Тонкий пластик за большие деньги. Возврат" → ["Цена / ценность"]
  Причина: здесь слово "возврат" просто обозначает решение вернуть товар, нет жалобы на сам возврат.

8. "Положительный / нейтральный отзыв"
   Положительный, нейтральный или информационный отзыв без явной жалобы.

Ставь этот label, если:

* отзыв положительный;
* отзыв нейтральный;
* покупатель просто сообщает факт без проблемы;
* товар понравился;
* все соответствует;
* отзыв без конкретной жалобы.

Важно:

* Если отзыв содержит позитивную часть, но есть конкретная проблема, не выбирай этот label.
* "Сам чемодан супер, но коробка в хлам" → не ставь "Положительный / нейтральный отзыв".

9. "Другая проблема"
   Есть негатив или жалоба, но она не подходит к классам выше.

Ставь этот label только если:

* есть негативная жалоба;
* невозможно уверенно отнести ее к качеству, размеру, несоответствию, упаковке, доставке, цене или возврату.

Важно:

* Не ставь "Другая проблема", если можно выбрать конкретный label.
* Не ставь "Другая проблема" вместе с конкретным проблемным label.
* "Ожидала лучше" без конкретики → "Другая проблема".
* "Да уж… лента не держится" → не "Другая проблема", а "Проблема с качеством товара".
* "Ерунда. Пузыри не получаются" → не "Другая проблема", а "Проблема с качеством товара".

Главные правила разграничения:

1. Не размечай по словам-триггерам.

* "Доставка" не всегда означает "Проблема доставки / получения".
* "Возврат" не всегда означает "Проблема с возвратом".
* "Деньги" не всегда означает "Цена / ценность".
* "Привезли" не всегда означает доставку.
* "Отправила обратно" не всегда означает проблему с возвратом.
* "Синтетика" не всегда означает "Проблема с качеством товара": если смысл в несоответствии заявленному/ожидаемому составу, это "Несоответствие карточке товара".

2. Качество vs несоответствие карточке.

* Товар плохой, сломан, хлипкий, не работает, не выполняет функцию → "Проблема с качеством товара".
* Товар отличается от фото, описания, заказанного варианта, заявленного материала, модели, цвета, состава или оригинальности → "Несоответствие карточке товара".
* Если есть и плохое качество, и явное несоответствие заявленному, ставь оба labels.
* Если материал отличается от ожидаемого/заявленного, по умолчанию ставь "Несоответствие карточке товара".
* Если материал отличается от ожидаемого/заявленного, но жалоба на качество материала является отдельной проблемой и причиной невозможности использовать/носить/оставить товар, добавь "Проблема с качеством товара".

3. Качество vs цена.

* "Товар плохой / бесполезный / не работает" → "Проблема с качеством товара".
* "Товар слишком плохой именно за такую цену / дорого / не стоит своих денег" → "Цена / ценность".
* Если при этом прямо названа проблема качества: "качество материала не дотягивает", "качество плохое", "качество могло быть лучше" — дополнительно ставь "Проблема с качеством товара".
* "Деньги на ветер" без жалобы на цену → обычно "Проблема с качеством товара".

4. Возврат.

* Ставь "Проблема с возвратом" только при жалобе на процесс возврата, деньги за возврат, платность возврата, удержание денег, отказ, компенсацию или процент выкупа.
* Не ставь этот label, если покупатель просто написал "возврат", "вернула", "отправила обратно".

5. Доставка.

* Ставь "Проблема доставки / получения" только при явной проблеме с доставкой, ПВЗ, курьером, складом, сроками, получением или неправильным вложением на этапе получения.
* Не ставь доставку только потому, что товар "пришел" или "привезли".

6. Упаковка и инструкция.

* Поврежденная коробка, мятая коробка, плохая упаковка, нарушенная упаковка, товар не подходит для подарка из-за коробки → "Проблема с комплектацией / упаковкой".
* Отсутствует товар/деталь/вложение → "Проблема с комплектацией / упаковкой".
* Отсутствует инструкция или инструкция есть, но она плохая/непонятная/мелкая/без картинок/без нормальных объяснений → "Проблема с комплектацией / упаковкой".
* Если покупатель связывает отсутствующее/неправильное вложение с ПВЗ, курьером, складом или службой доставки, добавь "Проблема доставки / получения".

Важные спорные случаи:

* "Сам чемодан супер!!! Но доставка 🤦🏼‍♀️ коробка в хлам …" → ["Проблема с комплектацией / упаковкой"]
  Причина: слово "доставка" здесь контекст, конкретная жалоба — коробка в плохом состоянии.

* "Ребят, ну явная подделка. Этикетки как попало наклеена, коробочка вся покоцанная. Раньше покупала эту пудру, было все нормально. Сейчас я в шоке, мел цветной. Пудра сама прям выпадает из футляра" → ["Проблема с качеством товара", "Проблема с комплектацией / упаковкой", "Несоответствие карточке товара"]
  Причина: плохое качество товара, поврежденная/плохая упаковка и явное подозрение на подделку.

* "В ПВЗ обнаружили, что ножниц то нет, вырезали их. Вот такая служба доставки" → ["Проблема доставки / получения", "Проблема с комплектацией / упаковкой"]
  Причина: отсутствует товар/вложение и покупатель связывает проблему со службой доставки/ПВЗ.

* "Хочу дополнить, может специально привозят не тот товар, чтоб снять процент выкупа" → ["Несоответствие карточке товара", "Проблема с возвратом"]
  Причина: речь о неправильном товаре и удержании/проценте выкупа.

* "Да уж… соплями реальней оклеить, чем этой лентой. Не держится нифига" → ["Проблема с качеством товара"]
  Причина: товар не выполняет базовую функцию.

* "Ужас! Бесполезная трата денег и времени ожидания!" → ["Проблема с качеством товара"]
  Причина: общий смысл — товар бесполезный/плохой; нет конкретной жалобы на цену или доставку.

* "Привезли не тот наполнитель. Заказывала 'Сенсацию свежести', привезли 'Классик'. Почему я из за вашей ошибки должна платить возврат. И в итоге стоимость наполнителя переплатили." → ["Несоответствие карточке товара", "Проблема с возвратом"]
  Причина: привезли другой вариант товара, плюс жалоба на оплату возврата; "Цена / ценность" лишняя, потому что речь не о завышенной цене, а о лишних расходах из-за ошибки.

* "Тонкий пластик за большие деньги. Возврат" → ["Цена / ценность"]
  Причина: основной смысл — слабый материал за высокую цену; "Возврат" здесь просто действие, а не жалоба на возврат; "Проблема с качеством товара" спорна и без явного дефекта не ставится.

* "Коробка в плачевном состоянии. Для подарка бы не подошла, а возврат почти 500₽" → ["Проблема с комплектацией / упаковкой", "Проблема с возвратом"]
  Причина: коробка повреждена, плюс есть жалоба на дорогой возврат.

* "Ерунда. Пузыри не получаются. Деньги на ветер" → ["Проблема с качеством товара"]
  Причина: товар не выполняет базовую функцию; "деньги на ветер" здесь не про цену.

* "Да, мне тоже привозили этих раздербаненных петухов. Отправила обратно. Просила курьера отметить, что неправильное вложение. Упаковка нарушена. Работники склада халатны." → ["Проблема с комплектацией / упаковкой", "Проблема доставки / получения", "Проблема с качеством товара"]
  Причина: товар в плохом состоянии, упаковка нарушена, проблема связана со складом/курьером/неправильным вложением.

* "Заказ пришел быстро, но оба флакона разбиты. Не понятно, за что сняли 100 руб. за возврат" → ["Проблема с качеством товара", "Проблема с возвратом"]
  Причина: товар разбит, плюс жалоба на списание денег за возврат.

* "Ткань тонкая, неприятная" → ["Проблема с качеством товара"]

* "На фото ткань плотная, а пришла тонкая" → ["Несоответствие карточке товара"]

* "Маломерит, но качество хорошее" → ["Проблема с размером / посадкой"]

* "Коробка порвана, но товар целый" → ["Проблема с комплектацией / упаковкой"]

* "Доставили поздно" → ["Проблема доставки / получения"]

* "Ожидала лучше" без конкретики → ["Другая проблема"]

* "Инструкция вообще непонятная, не хватает картинок и объяснений." → ["Проблема с комплектацией / упаковкой"]
  Причина: инструкция входит в комплект/оформление товара, и покупатель жалуется на ее качество.

* "Инструкция на коробке написана мелко и непонятно, пришлось разбираться долго." → ["Проблема с комплектацией / упаковкой"]
  Причина: есть проблема с инструкцией/объяснениями на коробке.

* "Ожидала хлопковую ткань, а пришла синтетика какая-то, ужасно неприятно на ощупь." → ["Несоответствие карточке товара"]
  Причина: главный смысл — материал не совпал с ожиданием/карточкой; качество отдельно не добавляем.

* "100% синтетика, хотя указано хлопок." → ["Несоответствие карточке товара"]
  Причина: прямо указано отличие фактического состава от заявленного.

* "... возврат потому, что 100% синтетика ... хотя указано хлопок ..." → ["Несоответствие карточке товара"]
  Причина: есть несоответствие заявленному составу; возврат здесь просто действие покупателя, а не жалоба на качество или процесс возврата.

* "Качество делало бы лучшего, но впринцепе за такую сумму." → ["Проблема с качеством товара"]
  Причина: есть прямая жалоба на качество; фраза "за такую сумму" здесь не является явной жалобой на цену.

* "Качество материала не дотягивает до заплаченной суммы, ожидал большего." → ["Проблема с качеством товара", "Цена / ценность"]
  Причина: есть прямая жалоба на качество материала и одновременно сравнение качества с заплаченной суммой.

Правила из ручной проверки, которые особенно важно соблюдать:

* Если в отзыве сказано, что привезли не тот товар, другой вариант, другой артикул, другой цвет, другой наполнитель или другой объем — ставь "Несоответствие карточке товара".
* Если вместе с неправильным товаром есть жалоба на платный возврат, процент выкупа, списание денег или платный отказ — дополнительно ставь "Проблема с возвратом".
* Если в ПВЗ обнаружили, что части товара нет, товар вырезали, украли, вскрыли или отсутствует вложение — ставь "Проблема с комплектацией / упаковкой". Если покупатель явно связывает это с ПВЗ/доставкой/службой доставки, дополнительно ставь "Проблема доставки / получения".
* Если товар разбит, сломан, поврежден, протек, треснул или пришел в непригодном состоянии — ставь "Проблема с качеством товара". Если при этом списали деньги за возврат или возврат платный — дополнительно ставь "Проблема с возвратом".
* Если жалоба только на плохую коробку, мятую упаковку или невозможность подарить из-за состояния упаковки — ставь "Проблема с комплектацией / упаковкой", но не ставь доставку без явной жалобы на срок, курьера, ПВЗ, склад или получение.
* Если покупатель пишет, что товар тонкий/хлипкий/плохой "за большие деньги" или "не стоит своих денег" — ставь "Цена / ценность". Не добавляй "Проблема с возвратом", если возврат упомянут только как решение вернуть товар без жалобы на деньги/отказ/платность возврата.
* Если покупатель жалуется на инструкцию, руководство, схему, объяснения на коробке, мелкий текст инструкции или отсутствие картинок/схем — ставь "Проблема с комплектацией / упаковкой".
* Если покупатель пишет, что ожидался/заявлен один материал, а пришел другой — ставь "Несоответствие карточке товара".
* Если материал не соответствует заявленному/ожидаемому, ставь "Несоответствие карточке товара". Сам факт возврата из-за несоответствия материала не добавляет "Проблема с качеством товара".
* Добавляй "Проблема с качеством товара" к несоответствию материала только при отдельной явной жалобе на качество материала: рвется, колется, сильный дискомфорт, плохая/хлипкая/непригодная ткань, качество материала не дотягивает.

Финальная самопроверка перед ответом:

* Не поставлен ли label только из-за слова-триггера?
* Не поставлена ли доставка только из-за слова "доставка", "привезли", "пришло"?
* Не поставлен ли возврат только из-за слова "возврат", "вернула", "отправила обратно"?
* Если в отзыве есть "платить возврат", "платный возврат", "дорогой возврат", "возврат почти ... рублей", "сняли деньги за возврат", "удержали деньги", "процент выкупа", "снять процент выкупа", "из-за вашей ошибки должна платить" — добавлена ли "Проблема с возвратом"?
* Если нет — добавь "Проблема с возвратом".
* Не поставлена ли цена только из-за слов "деньги", "деньги на ветер", "бесполезная трата денег"?
* Не перепутано ли плохое качество с несоответствием карточке?
* Если в отзыве есть "ожидала/думала/указано/заявлено X, а пришло Y", не перепутано ли это с качеством? Обычно это "Несоответствие карточке товара".
* Если жалоба на материал связана только с тем, что материал не совпал с ожиданием/описанием/карточкой, не добавлена ли ошибочно "Проблема с качеством товара"?
* Если материал не совпал с заявленным и из-за этого покупатель возвращает товар, не добавлена ли ошибочно "Проблема с качеством товара"? Сам возврат из-за несоответствия материала не является жалобой на качество.
* Если есть жалоба на инструкцию/руководство/схему/мелкий текст инструкции/нет картинок/непонятные объяснения, добавлена ли "Проблема с комплектацией / упаковкой"?
* Не выбран ли "Положительный / нейтральный отзыв" вместе с проблемой?
* Не выбран ли "Другая проблема" вместе с конкретным проблемным label?
* Если в отзыве есть "качество могло быть лучше", "качество не дотягивает", "качество материала не дотягивает", добавлена ли "Проблема с качеством товара"?
* Если материал не совпал с заявленным, не добавлена ли ошибочно "Проблема с качеством товара" только из-за возврата? Если нет отдельной жалобы на качество материала, оставь только "Несоответствие карточке товара".
* Если качество прямо сравнивается с ценой: "качество не соответствует цене", "качество материала не дотягивает до суммы", добавлены ли оба labels: "Проблема с качеством товара" и "Цена / ценность"?

Верни только JSON строго такого вида:
{
"items": [
{
"id": "тот же id",
"labels": ["один или несколько разрешенных labels"]
}
]
}
"""


In [16]:
RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "marketplace_review_relabeling_mvp",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "id": {"type": "string"},
                            "labels": {
                                "type": "array",
                                "items": {
                                    "type": "string",
                                    "enum": ALLOWED_LABELS_MVP,
                                },
                            },
                        },
                        "required": [
                            "id",
                            "labels",
                        ],
                    },
                }
            },
            "required": ["items"],
        },
    },
}


## 3. Загрузка трех CSV

Старые `labels` и `evidence` остаются в исходных файлах, но в итоговой разметке не используются как вход для модели.

В итоговом датафрейме будут сохранены служебные колонки:

- `source_file` — из какого файла пришел отзыв;
- `source_row_id` — номер строки внутри исходного файла;
- `global_id` — уникальный id для новой разметки.


In [17]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1251"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Не удалось прочитать файл {path}. Последняя ошибка: {last_error}")


def safe_filename(label: str) -> str:
    name = label.lower()
    name = name.replace(" / ", "_")
    name = name.replace("/", "_")
    name = name.replace(" ", "_")
    name = re.sub(r"[^а-яa-z0-9_]+", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name + ".csv"


def labels_to_list(labels: Any) -> list[str]:
    if isinstance(labels, list):
        raw = labels
    elif pd.isna(labels):
        raw = []
    elif isinstance(labels, str):
        value = labels.strip()
        if not value:
            raw = []
        else:
            try:
                parsed = json.loads(value)
                raw = parsed if isinstance(parsed, list) else []
            except Exception:
                if " | " in value:
                    raw = [x.strip() for x in value.split(" | ")]
                elif value in ALLOWED_LABELS_SET:
                    raw = [value]
                else:
                    raw = []
    else:
        raw = []

    result = []
    for label in raw:
        label = str(label).strip()
        if label in ALLOWED_LABELS_SET and label not in result:
            result.append(label)

    return result


def normalize_labels(labels: Any) -> list[str]:
    result = labels_to_list(labels)

    if not result:
        result = ["Другая проблема"]

    neutral_label = "Положительный / нейтральный отзыв"

    if neutral_label in result and len(result) > 1:
        result = [label for label in result if label != neutral_label]

    if not result:
        result = ["Другая проблема"]

    return result


def labels_to_json(labels: Any) -> str:
    return json.dumps(normalize_labels(labels), ensure_ascii=False)


def labels_to_str(labels: Any) -> str:
    return " | ".join(normalize_labels(labels))


def normalize_source_type(value: Any) -> str:
    source_type = str(value if not pd.isna(value) else "real").lower().strip()
    if source_type in ["synth", "synthetic", "generated"]:
        return "synthetic"
    if source_type not in ["real", "synthetic"]:
        return "real"
    return source_type


def load_input_by_class(input_dir: Path) -> pd.DataFrame:
    if not input_dir.exists():
        raise FileNotFoundError(f"Папка после синтетики не найдена: {input_dir}")

    records = {}
    summary_rows = []

    for class_label in ALLOWED_LABELS_MVP:
        path = input_dir / safe_filename(class_label)

        if not path.exists():
            print("Файл класса не найден, пропускаю:", path)
            summary_rows.append({"class_label": class_label, "file": str(path), "rows": 0, "status": "file_not_found"})
            continue

        part = read_csv_safely(path)

        if TEXT_COLUMN not in part.columns:
            raise ValueError(
                f"В файле {path} нет колонки {TEXT_COLUMN!r}. "
                f"Доступные колонки: {list(part.columns)}"
            )

        part = part.copy()
        part[TEXT_COLUMN] = part[TEXT_COLUMN].fillna("").astype(str).str.strip()
        part = part[part[TEXT_COLUMN].ne("")].copy()

        # В старой папке после синтетики основная колонка обычно называется labels.
        # Если old_labels уже есть, берем ее; иначе берем labels; затем добавляем class_label из имени файла.
        old_label_col = "old_labels" if "old_labels" in part.columns else "labels" if "labels" in part.columns else None

        if "source_type" not in part.columns:
            part["source_type"] = "real"

        for row_id, row in part.iterrows():
            text = str(row[TEXT_COLUMN]).strip()
            if not text:
                continue

            if text not in records:
                records[text] = {
                    TEXT_COLUMN: text,
                    "old_labels_set": [],
                    "source_type_set": set(),
                    "source_files": [],
                    "source_row_ids": [],
                }

            if old_label_col is not None:
                for label in normalize_labels(row.get(old_label_col, "")):
                    if label not in records[text]["old_labels_set"]:
                        records[text]["old_labels_set"].append(label)

            # Так как файл лежит в папке конкретного класса, сам class_label тоже считаем старой меткой.
            if class_label in ALLOWED_LABELS_SET and class_label not in records[text]["old_labels_set"]:
                records[text]["old_labels_set"].append(class_label)

            records[text]["source_type_set"].add(normalize_source_type(row.get("source_type", "real")))
            records[text]["source_files"].append(str(path.relative_to(PROJECT_ROOT)))
            records[text]["source_row_ids"].append(str(row_id))

        summary_rows.append({"class_label": class_label, "file": str(path.relative_to(PROJECT_ROOT)), "rows": len(part), "status": "ok"})

    rows = []
    for i, rec in enumerate(records.values()):
        source_type = "synthetic" if "synthetic" in rec["source_type_set"] else "real"
        old_labels = rec["old_labels_set"] or ["Другая проблема"]

        rows.append({
            "global_id": str(i),
            TEXT_COLUMN: rec[TEXT_COLUMN],
            OLD_LABELS_COLUMN: old_labels,
            "source_type": source_type,
            "source_files": " ; ".join(sorted(set(rec["source_files"]))),
            "source_row_ids": " ; ".join(rec["source_row_ids"]),
        })

    data = pd.DataFrame(rows)

    if data.empty:
        raise RuntimeError("Не удалось собрать входные данные из папки после синтетики.")

    if MAX_REVIEWS is not None:
        data = data.head(MAX_REVIEWS).copy()

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(OUTPUT_DIR / "input_by_class_summary.csv", index=False, encoding="utf-8-sig")

    return data.reset_index(drop=True)


df = load_input_by_class(INPUT_BY_CLASS_DIR)

print("Всего уникальных непустых отзывов после синтетики:", len(df))
print("Реальные / synthetic:")
print(df["source_type"].value_counts(dropna=False))
print()
print("Примеры старых labels:")
display(df[[TEXT_COLUMN, OLD_LABELS_COLUMN, "source_type", "source_files"]].head())


Всего уникальных непустых отзывов после синтетики: 1820
Реальные / synthetic:
source_type
real         1196
synthetic     624
Name: count, dtype: int64

Примеры старых labels:


,отзыв,old_labels,source_type,source_files
0,Не держится. Какая то пленка …,[Проблема с качеством товара],real,labeled/wb_feedbacks_by_class_with_synthetic/п...
1,"Куртка хорошая, качественная! Замок правда зли...",[Проблема с качеством товара],real,labeled/wb_feedbacks_by_class_with_synthetic/п...
2,"Задумка не плохая, но качество материала не по...","[Проблема с качеством товара, Проблема с разме...",real,labeled/wb_feedbacks_by_class_with_synthetic/п...
3,"Остаются темные потеки от конденсата,которые н...",[Проблема с качеством товара],real,labeled/wb_feedbacks_by_class_with_synthetic/п...
4,"Качество делало бы лучшего, но впринцепе за та...",[Проблема с качеством товара],real,labeled/wb_feedbacks_by_class_with_synthetic/п...


## 4. Вспомогательные функции для разметки

In [18]:
def load_done_ids(progress_path: Path) -> set[str]:
    if not progress_path.exists():
        return set()

    done = set()
    with progress_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
                done.add(str(obj["global_id"]))
            except Exception:
                continue

    return done


def append_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    with path.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def make_batches(data: pd.DataFrame, batch_size: int):
    for start in range(0, len(data), batch_size):
        yield data.iloc[start:start + batch_size]


def build_user_prompt(batch_df: pd.DataFrame) -> str:
    reviews = []

    for _, row in batch_df.iterrows():
        reviews.append({
            "id": str(row["global_id"]),
            "text": str(row[TEXT_COLUMN]),
        })

    return (
        "Ниже список отзывов. На входе только id и текст отзыва. "
        "Старые labels специально не передаются: разметь отзыв заново по смыслу. "
        "Верни JSON строго по схеме. "
        "Сохрани id каждого отзыва без изменений.\n\n"
        + json.dumps({"reviews": reviews}, ensure_ascii=False, indent=2)
    )


def validate_item(item: dict[str, Any]) -> dict[str, Any]:
    labels = item.get("labels", [])

    if not isinstance(labels, list):
        labels = ["Другая проблема"]

    labels = [str(label).strip() for label in labels]
    labels = [label for label in labels if label in ALLOWED_LABELS_SET]

    if not labels:
        labels = ["Другая проблема"]

    neutral_label = "Положительный / нейтральный отзыв"

    # Если есть конкретная проблема, нейтральный класс убираем.
    if neutral_label in labels and len(labels) > 1:
        labels = [label for label in labels if label != neutral_label]

    # Убираем дубли, сохраняя порядок.
    deduped = []
    for label in labels:
        if label not in deduped:
            deduped.append(label)

    return {
        "id": str(item.get("id", "")),
        "labels": deduped,
    }


def label_batch(batch_df: pd.DataFrame, max_retries: int = 3) -> list[dict[str, Any]]:
    user_prompt = build_user_prompt(batch_df)
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            print(
                f"Отправляю батч в OpenAI API. "
                f"Попытка {attempt}/{max_retries}. "
                f"Размер батча: {len(batch_df)}. "
                f"MODEL={MODEL}. "
                f"MAX_COMPLETION_TOKENS={MAX_COMPLETION_TOKENS}"
            )

            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                response_format=RESPONSE_FORMAT,

                timeout=300,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )

            finish_reason = response.choices[0].finish_reason
            content = response.choices[0].message.content or ""

            if finish_reason != "stop":
                debug_path = OUTPUT_DIR / "debug_length_response.txt"
                with debug_path.open("w", encoding="utf-8") as f:
                    f.write(content)
                raise RuntimeError(
                    f"Ответ модели был обрезан. finish_reason={finish_reason}. "
                    f"Частичный ответ сохранен в {debug_path}"
                )

            try:
                parsed = json.loads(content)
            except json.JSONDecodeError as e:
                debug_path = OUTPUT_DIR / "debug_bad_openai_response.txt"
                with debug_path.open("w", encoding="utf-8") as f:
                    f.write(content)

                raise RuntimeError(
                    f"Модель вернула невалидный JSON. "
                    f"Ответ сохранен в {debug_path}. "
                    f"Ошибка: {e}"
                )

            raw_items = parsed.get("items", [])
            validated = [validate_item(item) for item in raw_items]

            expected_ids = set(batch_df["global_id"].astype(str))
            returned_ids = {item["id"] for item in validated}

            missing_ids = expected_ids - returned_ids
            if missing_ids:
                for missing_id in missing_ids:
                    validated.append({
                        "id": missing_id,
                        "labels": ["Другая проблема"],
                    })

            return validated

        except Exception as e:
            last_error = e
            wait = max(20, 2 ** attempt)
            print(f"Ошибка на попытке {attempt}/{max_retries}: {e}")
            print(f"Ждем {wait} сек. и пробуем снова...")
            time.sleep(wait)

    # Если батч больше 1 и не прошел, делим его на части.
    if len(batch_df) > 1:
        mid = len(batch_df) // 2
        left = batch_df.iloc[:mid].copy()
        right = batch_df.iloc[mid:].copy()
        return label_batch(left, max_retries=max_retries) + label_batch(right, max_retries=max_retries)

    # Если даже один отзыв не удалось разметить, сохраняем его как "Другая проблема".
    row = batch_df.iloc[0]
    return [{
        "id": str(row["global_id"]),
        "labels": ["Другая проблема"],
    }]


## 5. Тестовая разметка маленького батча

Перед полным запуском проверь качество на 5 отзывах.


In [19]:
test_batch = df.head(min(5, len(df))).copy()
test_result = label_batch(test_batch)

test_result_df = pd.DataFrame(test_result).rename(columns={"labels": NEW_LABELS_COLUMN})
test_result_df = test_batch[["global_id", TEXT_COLUMN, OLD_LABELS_COLUMN]].merge(
    test_result_df,
    left_on="global_id",
    right_on="id",
    how="left",
)
test_result_df = test_result_df[[TEXT_COLUMN, OLD_LABELS_COLUMN, NEW_LABELS_COLUMN]]
test_result_df


Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 5. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000


,отзыв,old_labels,new_labels
0,Не держится. Какая то пленка …,[Проблема с качеством товара],[Проблема с качеством товара]
1,"Куртка хорошая, качественная! Замок правда зли...",[Проблема с качеством товара],[Проблема с качеством товара]
2,"Задумка не плохая, но качество материала не по...","[Проблема с качеством товара, Проблема с разме...",[Проблема с качеством товара]
3,"Остаются темные потеки от конденсата,которые н...",[Проблема с качеством товара],[Проблема с качеством товара]
4,"Качество делало бы лучшего, но впринцепе за та...",[Проблема с качеством товара],[Проблема с качеством товара]


## 6. Основная разметка

После каждого батча результат сохраняется в:

```text
labeled/wb_feedbacks_ChatGpt_markup_from_synthetic_gpt5/chatgpt_labeled_reviews_mvp_progress.jsonl
```

Если ноутбук упадет или остановится, можно запустить заново — уже размеченные `global_id` будут пропущены.


In [20]:
done_ids = load_done_ids(COMBINED_PROGRESS_JSONL_PATH)
print("Уже размечено:", len(done_ids))

todo_df = df[~df["global_id"].astype(str).isin(done_ids)].copy()
print("Осталось разметить:", len(todo_df))

batches = list(make_batches(todo_df, BATCH_SIZE))

for batch_df in tqdm(batches, desc="Разметка батчей"):
    labeled_items = label_batch(batch_df)

    meta_by_id = {
        str(row["global_id"]): {
            "global_id": str(row["global_id"]),
            TEXT_COLUMN: str(row[TEXT_COLUMN]),
            OLD_LABELS_COLUMN: row[OLD_LABELS_COLUMN],
            "source_type": str(row["source_type"]),
            "source_files": str(row["source_files"]),
            "source_row_ids": str(row["source_row_ids"]),
        }
        for _, row in batch_df.iterrows()
    }

    rows_to_save = []

    for item in labeled_items:
        global_id = str(item["id"])
        meta = meta_by_id.get(global_id, {})
        new_labels = item["labels"]

        rows_to_save.append({
            "global_id": global_id,
            TEXT_COLUMN: meta.get(TEXT_COLUMN, ""),
            OLD_LABELS_COLUMN: meta.get(OLD_LABELS_COLUMN, []),
            NEW_LABELS_COLUMN: new_labels,
            # labels оставляем для совместимости со старыми ноутбуками; это те же new_labels.
            "labels": new_labels,
            "source_type": meta.get("source_type", ""),
            "source_files": meta.get("source_files", ""),
            "source_row_ids": meta.get("source_row_ids", ""),
        })

    append_jsonl(COMBINED_PROGRESS_JSONL_PATH, rows_to_save)
    time.sleep(SLEEP_SECONDS)

print("Готово.")


Уже размечено: 120
Осталось разметить: 1700


Разметка батчей:   0%|          | 0/850 [00:00<?, ?it/s]

Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 2. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 2. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 2. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 2. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 2. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 2. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 2. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 2. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 2. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 2. MODEL=gpt-5. MAX_COMPLETION_TOKENS=2000
Отправляю 

## 7. Сбор итогового combined-файла

In [21]:
def read_progress_jsonl(path: Path) -> pd.DataFrame:
    rows = []

    if not path.exists():
        return pd.DataFrame()

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return pd.DataFrame(rows)


def make_output(data: pd.DataFrame) -> pd.DataFrame:
    cols = [
        TEXT_COLUMN,
        OLD_LABELS_COLUMN,
        NEW_LABELS_COLUMN,
        "labels",
        "source_type",
        "source_files",
        "source_row_ids",
    ]

    output = data[[col for col in cols if col in data.columns]].copy()
    output[OLD_LABELS_COLUMN] = output[OLD_LABELS_COLUMN].apply(labels_to_json)
    output[NEW_LABELS_COLUMN] = output[NEW_LABELS_COLUMN].apply(labels_to_json)
    output["labels"] = output[NEW_LABELS_COLUMN]  # совместимость: labels = new_labels
    output["old_labels_str"] = output[OLD_LABELS_COLUMN].apply(labels_to_str)
    output["new_labels_str"] = output[NEW_LABELS_COLUMN].apply(labels_to_str)
    return output


labeled_df = read_progress_jsonl(COMBINED_PROGRESS_JSONL_PATH)

if labeled_df.empty:
    raise RuntimeError("Файл прогресса пустой. Сначала запусти разметку.")

labeled_df = labeled_df.drop_duplicates(subset=["global_id"], keep="last").reset_index(drop=True)

combined_output_df = make_output(labeled_df)
combined_output_df.to_csv(COMBINED_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Сохранено:", COMBINED_OUTPUT_PATH)
print("Размер:", combined_output_df.shape)

combined_output_df.head()


Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_ChatGpt_markup_from_synthetic_gpt5_V_2/chatgpt_labeled_reviews_mvp_combined.csv
Размер: (1820, 9)


,отзыв,old_labels,new_labels,labels,source_type,source_files,source_row_ids,old_labels_str,new_labels_str
0,Не держится. Какая то пленка …,"[""Проблема с качеством товара""]","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,0,Проблема с качеством товара,Проблема с качеством товара
1,"Куртка хорошая, качественная! Замок правда зли...","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,1,Проблема с качеством товара,Проблема с качеством товара
2,"Задумка не плохая, но качество материала не по...","[""Проблема с качеством товара"", ""Проблема с ра...","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,2 ; 4,Проблема с качеством товара | Проблема с разме...,Проблема с качеством товара
3,"Остаются темные потеки от конденсата,которые н...","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,3,Проблема с качеством товара,Проблема с качеством товара
4,"Качество делало бы лучшего, но впринцепе за та...","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,4,Проблема с качеством товара,Проблема с качеством товара


## 8. Сохранение отдельных файлов по источникам

Дополнительно ноутбук сохраняет отдельные CSV для каждого исходного файла.


In [22]:
import shutil

# Пересобираем папку по классам на основе new_labels.
# Старая папка с синтетикой не трогается.
if BY_CLASS_OUTPUT_DIR.exists():
    shutil.rmtree(BY_CLASS_OUTPUT_DIR)
BY_CLASS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

by_class_summary = []

for label in ALLOWED_LABELS_MVP:
    mask = labeled_df[NEW_LABELS_COLUMN].apply(lambda labels: label in normalize_labels(labels))
    part = labeled_df[mask].copy()

    output_path = BY_CLASS_OUTPUT_DIR / safe_filename(label)

    if part.empty:
        part_output = pd.DataFrame(columns=[
            TEXT_COLUMN,
            OLD_LABELS_COLUMN,
            NEW_LABELS_COLUMN,
            "labels",
            "old_labels_str",
            "new_labels_str",
            "source_type",
            "source_files",
            "source_row_ids",
        ])
    else:
        part_output = make_output(part)

    part_output.to_csv(output_path, index=False, encoding="utf-8-sig")

    by_class_summary.append({
        "label": label,
        "file": str(output_path.relative_to(PROJECT_ROOT)),
        "rows": len(part_output),
        "real_rows": int((part_output.get("source_type", pd.Series(dtype=str)) == "real").sum()) if not part_output.empty else 0,
        "synthetic_rows": int((part_output.get("source_type", pd.Series(dtype=str)) == "synthetic").sum()) if not part_output.empty else 0,
    })

    print("Сохранено:", output_path, "| строк:", len(part_output))

by_class_summary_df = pd.DataFrame(by_class_summary)
by_class_summary_df.to_csv(OUTPUT_DIR / "by_class_output_summary.csv", index=False, encoding="utf-8-sig")
by_class_summary_df


Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic_gpt5_relabelled_V_2/проблема_с_качеством_товара.csv | строк: 634
Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic_gpt5_relabelled_V_2/проблема_с_размером_посадкой.csv | строк: 220
Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic_gpt5_relabelled_V_2/несоответствие_карточке_товара.csv | строк: 251
Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic_gpt5_relabelled_V_2/проблема_с_комплектацией_упаковкой.csv | строк: 437
Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic_gpt5_relabelled_V_2/проблема_доставки_получения.csv | строк: 211
Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic_gpt5_relabelled_V_2/цена_цен

,label,file,rows,real_rows,synthetic_rows
0,Проблема с качеством товара,labeled/wb_feedbacks_by_class_with_synthetic_g...,634,524,110
1,Проблема с размером / посадкой,labeled/wb_feedbacks_by_class_with_synthetic_g...,220,141,79
2,Несоответствие карточке товара,labeled/wb_feedbacks_by_class_with_synthetic_g...,251,223,28
3,Проблема с комплектацией / упаковкой,labeled/wb_feedbacks_by_class_with_synthetic_g...,437,370,67
4,Проблема доставки / получения,labeled/wb_feedbacks_by_class_with_synthetic_g...,211,84,127
5,Цена / ценность,labeled/wb_feedbacks_by_class_with_synthetic_g...,188,38,150
6,Проблема с возвратом,labeled/wb_feedbacks_by_class_with_synthetic_g...,249,231,18
7,Положительный / нейтральный отзыв,labeled/wb_feedbacks_by_class_with_synthetic_g...,211,80,131
8,Другая проблема,labeled/wb_feedbacks_by_class_with_synthetic_g...,69,16,53


## 9. Проверка распределения новых классов

In [23]:
from collections import Counter

counter = Counter()

for labels in labeled_df[NEW_LABELS_COLUMN]:
    for label in normalize_labels(labels):
        counter[label] += 1

stats_df = (
    pd.DataFrame(counter.items(), columns=["label", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

stats_df["percent_of_reviews"] = (stats_df["count"] / len(labeled_df) * 100).round(2)
stats_df


,label,count,percent_of_reviews
0,Проблема с качеством товара,634,34.84
1,Проблема с комплектацией / упаковкой,437,24.01
2,Несоответствие карточке товара,251,13.79
3,Проблема с возвратом,249,13.68
4,Проблема с размером / посадкой,220,12.09
5,Проблема доставки / получения,211,11.59
6,Положительный / нейтральный отзыв,211,11.59
7,Цена / ценность,188,10.33
8,Другая проблема,69,3.79


## 10. Примеры на ручную проверку

Так как итоговый вывод теперь содержит только `отзыв` и `labels`, отдельные поля `confidence`, `evidence` и `needs_review` не используются.

Для ручной проверки сохраняются отзывы с классом `Другая проблема`.


In [24]:
needs_review_df = labeled_df[
    labeled_df[NEW_LABELS_COLUMN].apply(lambda labels: "Другая проблема" in normalize_labels(labels))
].copy()

needs_review_output_df = make_output(needs_review_df)
needs_review_output_df.to_csv(NEEDS_REVIEW_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("На ручную проверку:", len(needs_review_output_df))
print("Сохранено:", NEEDS_REVIEW_OUTPUT_PATH)

needs_review_output_df.head(30)


На ручную проверку: 69
Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_ChatGpt_markup_from_synthetic_gpt5_V_2/chatgpt_labeled_reviews_mvp_needs_review.csv


,отзыв,old_labels,new_labels,labels,source_type,source_files,source_row_ids,old_labels_str,new_labels_str
45,Не понравился материал и фасон. Отказ,"[""Проблема с качеством товара""]","[""Другая проблема""]","[""Другая проблема""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,45,Проблема с качеством товара,Другая проблема
80,Сумка непонравилась,"[""Проблема с качеством товара""]","[""Другая проблема""]","[""Другая проблема""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,80,Проблема с качеством товара,Другая проблема
109,Не понравилась ткань.,"[""Проблема с качеством товара""]","[""Другая проблема""]","[""Другая проблема""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,109,Проблема с качеством товара,Другая проблема
124,не понравился материал,"[""Проблема с качеством товара""]","[""Другая проблема""]","[""Другая проблема""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,124,Проблема с качеством товара,Другая проблема
483,Орехи в тесте. Фу!!!,"[""Проблема с качеством товара""]","[""Другая проблема""]","[""Другая проблема""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,483,Проблема с качеством товара,Другая проблема
1478,"Хорошая поилка, но мыть заколебешься","[""Положительный / нейтральный отзыв""]","[""Другая проблема""]","[""Другая проблема""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,57,Положительный / нейтральный отзыв,Другая проблема
1483,Скромненько. Соответствует цене. Отказ.,"[""Положительный / нейтральный отзыв""]","[""Другая проблема""]","[""Другая проблема""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,62,Положительный / нейтральный отзыв,Другая проблема
1621,Больше брать не буду,"[""Другая проблема""]","[""Другая проблема""]","[""Другая проблема""]",real,labeled/wb_feedbacks_by_class_with_synthetic/д...,0,Другая проблема,Другая проблема
1622,"всего месяц остался до конца срока годности, н...","[""Другая проблема""]","[""Другая проблема""]","[""Другая проблема""]",real,labeled/wb_feedbacks_by_class_with_synthetic/д...,1,Другая проблема,Другая проблема
1624,Костюм не понравился,"[""Другая проблема""]","[""Другая проблема""]","[""Другая проблема""]",real,labeled/wb_feedbacks_by_class_with_synthetic/д...,3,Другая проблема,Другая проблема


## 11. One-hot файл для обучения классификатора

Создает бинарную колонку для каждого нового класса.


In [25]:
train_df = make_output(labeled_df)
train_df.to_csv(TRAIN_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Файл для обучения сохранен:", TRAIN_OUTPUT_PATH)
train_df.head()


Файл для обучения сохранен: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_ChatGpt_markup_from_synthetic_gpt5_V_2/chatgpt_labeled_reviews_mvp_for_training.csv


,отзыв,old_labels,new_labels,labels,source_type,source_files,source_row_ids,old_labels_str,new_labels_str
0,Не держится. Какая то пленка …,"[""Проблема с качеством товара""]","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,0,Проблема с качеством товара,Проблема с качеством товара
1,"Куртка хорошая, качественная! Замок правда зли...","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,1,Проблема с качеством товара,Проблема с качеством товара
2,"Задумка не плохая, но качество материала не по...","[""Проблема с качеством товара"", ""Проблема с ра...","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,2 ; 4,Проблема с качеством товара | Проблема с разме...,Проблема с качеством товара
3,"Остаются темные потеки от конденсата,которые н...","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,3,Проблема с качеством товара,Проблема с качеством товара
4,"Качество делало бы лучшего, но впринцепе за та...","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]","[""Проблема с качеством товара""]",real,labeled/wb_feedbacks_by_class_with_synthetic/п...,4,Проблема с качеством товара,Проблема с качеством товара


## 12. Проверка случайных примеров по каждому классу

In [26]:
for label in ALLOWED_LABELS_MVP:
    print("\n" + "=" * 100)
    print(label)
    print("=" * 100)

    mask = labeled_df[NEW_LABELS_COLUMN].apply(lambda labels: label in normalize_labels(labels))
    examples = labeled_df[mask].sample(
        n=min(5, int(mask.sum())),
        random_state=42,
    ) if mask.any() else pd.DataFrame()

    for _, row in examples.iterrows():
        print("- Отзыв:", row[TEXT_COLUMN])
        print("  Old labels:", labels_to_str(row[OLD_LABELS_COLUMN]))
        print("  New labels:", labels_to_str(row[NEW_LABELS_COLUMN]))
        print()



Проблема с качеством товара
- Отзыв: товар пришёл с браком, на фото видно, такое нельзя давать малышу, Фото сделано прям в пункте выдачи и, где и попросила оформить возврат без снятия денег за возврат, оформить по браку, так же я заказывала с олной прорезью а пришло не то, пришло с четырьмя, ещё и острые края поло маны и выгнуты. прошу не снимать денег за возврат
  Old labels: Проблема с качеством товара | Несоответствие карточке товара | Проблема с возвратом
  New labels: Проблема с качеством товара | Несоответствие карточке товара | Проблема с возвратом

- Отзыв: Банка пришла с дефектом, пропитка частично вытекла. Вернул обратно в пункте выдачи.
  Old labels: Проблема с качеством товара
  New labels: Проблема с качеством товара

- Отзыв: Пришли совсем без упаковки. Полностью разбитые. Сняли деньги за отказ. Верните деньги и научитесь упаковывать. Ужас
  Old labels: Проблема с качеством товара | Проблема с комплектацией / упаковкой | Проблема с возвратом
  New labels: Проблема с каче